# Recovery test beyond maths: does a model fix a wrong step 2 outside maths?

The model's answer already holds steps 1 and 2, and the model writes the rest (greedy, thinking off). Each start comes in four versions, as in the 14 Nov wait check: step 2 wrong or correct, each with and without "Wait" after step 2.

- **GSM8K-SC**: the 60 maths problems of the wait check, as the reference.
- **ProcBench**: seven procedures on strings and lists, 20 problems each, with step 2 edited by a script.
- **BIG-Bench Mistake**: PaLM 2's own first mistake in step 2 (word sorting, tracking shuffled objects, logical deduction, and multistep arithmetic as a maths bridge).

Each model makes about 1,360 continuations. Models: Qwen3-8B (the wait check's model) and Shree's three Olmo 3 7B Instruct checkpoints (SFT, DPO, and the final model after RL).

Kaggle settings: **Accelerator: GPU T4 x2** and **Internet: on**. The script splits each model across both GPUs in bfloat16, which a T4 emulates. Downloads are 16 GB for Qwen3-8B and 15 GB for each Olmo model; the loop deletes each model after its run. The test has not been timed on a T4 yet. The log prints new tokens per second after every batch, and a run that stops can be restarted: it skips what `generations.jsonl` already holds.

On Colab's single T4, add `--load-4bit` (and run `pip install bitsandbytes` first).

In [ ]:
REPO_URL = "https://github.com/amitkov-sudo/grpo-vs-gfn-self-correction.git"

!git clone -q $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt
# Kaggle ships torchao 0.10, which recent PEFT rejects, and this repository does not use torchao.
!pip uninstall -q -y torchao

In [ ]:
!python -m pytest -q tests
!python eval/recovery_test.py --smoke

In [ ]:
import os, shutil

MODELS = [
    "Qwen/Qwen3-8B",                    # the wait check's model
    "allenai/Olmo-3-7B-Instruct-SFT",   # Shree's checkpoints
    "allenai/Olmo-3-7B-Instruct-DPO",
    "allenai/Olmo-3-7B-Instruct",       # the final model, after RL
]
EXTRA = ""   # "--dtype float16" runs faster on a T4 but can overflow; the script stops if the first batch is garbled
HUB = os.path.expanduser("~/.cache/huggingface/hub")

for m in MODELS:
    !python eval/recovery_test.py --model {m} {EXTRA}
    shutil.rmtree(os.path.join(HUB, "models--" + m.replace("/", "--")), ignore_errors=True)   # free the disk

In [ ]:
import glob, json
import pandas as pd

CONDS = ["wrong", "wrong_wait", "clean", "clean_wait"]
rows = []
for path in sorted(glob.glob("outputs/recovery-*/summary.json")):
    s = json.load(open(path))
    for group, by in s["groups"].items():
        if not group.startswith("  "):
            rows.append([group, s["config"]["model"]] +
                        [f"{by[c]['right']}/{by[c]['n']}" if c in by else "-" for c in CONDS])
table = pd.DataFrame(rows, columns=["group", "model"] + CONDS).sort_values(["group", "model"])
print("Right final answers. wrong = step 2 wrong; _wait = 'Wait' after step 2; clean = step 2 correct.")
display(table)
for path in sorted(glob.glob("outputs/recovery-*/summary.txt")):
    print(open(path).read())